In [1]:
#Compruebo la GPU y el disco
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /workspace

name, memory.total [MiB]
NVIDIA H100 80GB HBM3, 81559 MiB
Filesystem      Size  Used Avail Use% Mounted on
overlay         100G   16M  100G   1% /


In [2]:
import os

os.environ["HF_HOME"] = "/workspace/hf"
os.makedirs("/workspace/resultados", exist_ok=True)

#El token viene del secreto de RunPod
print("HF_TOKEN configurado:", "HF_TOKEN" in os.environ)

HF_TOKEN configurado: True


In [ ]:
%cd /workspace
# vLLM, la misma versión que en el entrenamiento
!pip install -q uv pandas transformers
!uv venv vllm --python 3.12
!uv pip install --python vllm/bin/python "vllm==0.33.0"

In [ ]:
#FlashInfer kernels are disabled fix
#Si se queda atascado más de 5 min (red lenta), parar la celda: a temperatura 0 no hace falta (ver arrancar)
!./vllm/bin/vllm download-kernels

In [6]:
from huggingface_hub import HfApi

#Subidas a mi repo de checkpoints: copio el hash del epoch 2 (la última) y, si quiero, el del epoch 0
for commit in HfApi().list_repo_commits("gabmerlo/salamandra-7b-eagle3-alia"):
    print(commit.created_at.strftime("%H:%M"), commit.commit_id, commit.title)

19:19 a6e162549090dec937efd3f4b58460e51098ada4 Upload folder using huggingface_hub
18:59 4b1ab1537349ac1b1b52dc1a073f13ab17494397 Upload folder using huggingface_hub
17:38 381ed00bab0e036b3e5b977a608afab877e3a749 Upload folder using huggingface_hub
17:12 830f1fa96c190bbea10e85866e439dd342e0bb4d Upload folder using huggingface_hub
16:50 a05f3127df3f93a74a06fbe667aebbdf521b589f Upload folder using huggingface_hub
16:50 de20c4c3e37a17fad9fbbdfbaea6f2a09e8c18b2 initial commit


In [40]:
#Fechas de subida para elegir la versión del 7B instruct de la época del draft de IBM
for repo in ["BSC-LT/salamandra-2b-instruct-gptq", "BSC-LT/salamandra-7b-instruct"]:
    print("==", repo)
    for c in HfApi().list_repo_commits(repo):
        print(c.created_at.strftime("%Y-%m-%d"), c.commit_id, c.title)

== BSC-LT/salamandra-2b-instruct-gptq
2025-03-26 ba74d96c7bb72c62b73c3e04f7342a4e9cccfffb Merging version 1.1
2025-03-18 6416025fe4e07ac02bb01d0c7414ed7ebff0a486 Upload LlamaForCausalLM
2025-03-18 70b8f95fe77a01c0d083061b109a70030cbf7d46 Upload tokenizer
2025-03-18 c5b73f126b6a2a0ca1c3ed33a048c3effd3157bb Empty branch
2024-11-18 c6a50199d9062646c72c86f3edf274352f0f4231 Update README.md
2024-11-18 9350915e13dd658b56feabc22c7d694bb4efb74a Update README.md
2024-11-18 0e07159eee369fa0c59ebdb9e727d82f86d13ff6 Upload LlamaForCausalLM
2024-11-18 ce92423987e23338d48ea145d22c6ef7a0fceaa5 Upload tokenizer
2024-11-18 c0b47c95d06beed3c6586fb3a127dfce638ce77b Delete tokenizer_config.json
2024-11-18 09201ef2a7ad6d0687cdf32c2c8d3f1683276449 Delete tokenizer.json
2024-11-18 ecf99df529501f6b7431fee9c4cc2216d795e03d Delete special_tokens_map.json
2024-11-18 72490fad1dbf222f4fa1b70269b13dd8eb66f059 Delete model.safetensors
2024-11-18 df72f4f7e7d81c99ca798837335d47d9c1481c16 Delete generation_config.json


In [42]:
#Configuraciones a comparar. Para no medir alguna, comentar su línea
N_ESPECULATIVOS = 3   #tokens que propone el draft por paso (EAGLE-3 entrenado con 3)
REV_7B_V1 = "0be4f2d2a9f859ddad31b35e3f13059944916644"   #versión del 7B instruct antiguo de la época del draft de IBM

OBJETIVO = "BSC-LT/salamandra-7b-instruct-2606"

CONFIGURACIONES = {
    #Sobre el 2606 (ya medidas, no se repiten)
    "base": {"tipo": "base"},
    "eagle3_ep2": {"tipo": "eagle", "revision": "a6e162549090dec937efd3f4b58460e51098ada4"},
    "draft_2b_base_gptq": {"tipo": "draft", "modelo": "BSC-LT/salamandra-2b-base-gptq"},
    #Nuevo: el draft instruct de IBM sobre el 2606 (lo que hay disponible hoy)
    "draft_2b_instruct_gptq": {"tipo": "draft", "modelo": "BSC-LT/salamandra-2b-instruct-gptq"},
    #Nuevo: cada draft con el modelo para el que se hizo (7B instruct antiguo)
    "base_7b_instruct_v1": {"tipo": "base", "objetivo": "BSC-LT/salamandra-7b-instruct",
                            "revision_objetivo": REV_7B_V1},
    "draft_2b_instruct_gptq_v1": {"tipo": "draft", "modelo": "BSC-LT/salamandra-2b-instruct-gptq",
                                  "objetivo": "BSC-LT/salamandra-7b-instruct",
                                  "revision_objetivo": REV_7B_V1, "referencia": "base_7b_instruct_v1"},
}

In [43]:
#Bajo todo antes de medir: los modelos objetivo, mis checkpoints (sin el optimizador) y los drafts del BSC
!./vllm/bin/hf download {OBJETIVO} > /dev/null

for nombre, config in CONFIGURACIONES.items():
    revision = config.get("revision")
    modelo_draft = config.get("modelo")
    objetivo = config.get("objetivo")
    revision_objetivo = config.get("revision_objetivo", "main")
    if objetivo:
        !./vllm/bin/hf download {objetivo} --revision {revision_objetivo} > /dev/null
    if config["tipo"] == "eagle":
        if not os.path.exists(f"/workspace/{nombre}/config.json"):
            !rm -rf /workspace/{nombre}
            !./vllm/bin/hf download gabmerlo/salamandra-7b-eagle3-alia --revision {revision} --local-dir /workspace/{nombre} --exclude "optimizer_state_dict.pt" --exclude "scheduler_state_dict.pt" --exclude "train.log" > /dev/null
    elif config["tipo"] == "draft":
        !./vllm/bin/hf download {modelo_draft} > /dev/null
    print("Descargado:", nombre)

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 16 files: 100%|█████████████████████| 16/16 [00:00<00:00, 1413.80it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Descargado: base
Descargado: eagle3_ep2
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 1342.27it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Descargado: draft_2b_base_gptq
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files:   0%|                       

In [44]:
from transformers import AutoTokenizer

tokenizador_objetivo = AutoTokenizer.from_pretrained(OBJETIVO)

#vLLM pasa al draft los IDs del objetivo: hace falta el mismo vocabulario
#(solo se toleran cambios de nombre en los tokens reservados 6-13, que no salen en estas prompts)
for nombre, config in list(CONFIGURACIONES.items()):
    if config["tipo"] != "draft":
        continue
    tok_obj = AutoTokenizer.from_pretrained(config.get("objetivo", OBJETIVO), revision=config.get("revision_objetivo", "main"))
    tok_draft = AutoTokenizer.from_pretrained(config["modelo"])
    v_obj, v_draft = tok_obj.get_vocab(), tok_draft.get_vocab()
    distintos = [t for t in set(v_obj) | set(v_draft) if v_obj.get(t) != v_draft.get(t)]
    ids = ({v_obj.get(t) for t in distintos} | {v_draft.get(t) for t in distintos}) - {None}
    compatible = len(v_obj) == len(v_draft) and ids <= set(range(6, 14))
    print(nombre, "| tokens distintos:", len(distintos), "| compatible:", compatible)
    if not compatible:
        print("   → lo quito de la comparación. IDs distintos:", sorted(ids)[:20])
        del CONFIGURACIONES[nombre]

draft_2b_base_gptq | tokens distintos: 16 | compatible: True
draft_2b_instruct_gptq | tokens distintos: 16 | compatible: True
draft_2b_instruct_gptq_v1 | tokens distintos: 0 | compatible: True


In [ ]:
import json
from collections import Counter

N_POR_IDIOMA = 20
MAX_TOKENS = 512

def prompt_de(conversacion):
    #Me quedo con los mensajes hasta la primera respuesta del asistente
    mensajes = []
    for mensaje in conversacion["messages"]:
        if mensaje["role"] == "assistant":
            break
        mensajes.append({"role": mensaje["role"], "content": mensaje["content"]})
    return mensajes


{'es': 20, 'ca': 20, 'eu': 20, 'gl': 20}


In [45]:
import subprocess, signal, time, re
import requests

URL = "http://127.0.0.1:8000"

def argumentos(nombre, config):
    #Cómo arranca vLLM cada configuración
    comunes = ["--max-model-len", "8192", "--port", "8000", "--no-enable-prefix-caching"]
    objetivo = config.get("objetivo", OBJETIVO)
    if "revision_objetivo" in config:
        comunes += ["--revision", config["revision_objetivo"], "--tokenizer-revision", config["revision_objetivo"]]
    if config["tipo"] == "base":
        return [objetivo] + comunes
    if config["tipo"] == "eagle":
        return ["/workspace/" + nombre] + comunes
    especulativo = json.dumps({"model": config["modelo"], "num_speculative_tokens": N_ESPECULATIVOS})
    return [objetivo, "--speculative-config", especulativo] + comunes

def arrancar(nombre, config):
    try:
        requests.get(URL + "/v1/models", timeout=2)
        raise RuntimeError("Ya hay un servidor en el puerto 8000: ejecuta !pkill -f \"vllm serve\"")
    except requests.exceptions.ConnectionError:
        pass
    log = "/workspace/vllm_" + nombre + ".log"
    #Sin el sampler de FlashInfer (a temperatura 0 no hace falta) y con ninja en el PATH por si acaso
    entorno = {**os.environ, "VLLM_USE_FLASHINFER_SAMPLER": "0", "PATH": "/workspace/vllm/bin:" + os.environ["PATH"]}

    with open(log, "w") as archivo:
        servidor = subprocess.Popen(
            ["./vllm/bin/vllm", "serve"] + argumentos(nombre, config),
            stdout=archivo,
            stderr=subprocess.STDOUT,
            env=entorno,
            start_new_session=True
        )

    #Espero a que responda
    while True:
        if servidor.poll() is not None:
            raise RuntimeError("vLLM se ha parado, mirar " + log)
        try:
            requests.get(URL + "/v1/models", timeout=5)
            break
        except Exception:
            time.sleep(10)

    modelo = requests.get(URL + "/v1/models").json()["data"][0]["id"]
    print("vLLM listo:", nombre)
    return servidor, modelo

def parar(servidor):
    os.killpg(servidor.pid, signal.SIGTERM)
    servidor.wait()
    time.sleep(15)

def metricas_spec():
    #Contadores de speculative decoding de vLLM (solo existen con draft)
    valores = {}
    for linea in requests.get(URL + "/metrics").text.splitlines():
        if not linea.startswith("vllm:spec_decode") or "_created" in linea:
            continue
        nombre_completo, valor = linea.rsplit(" ", 1)
        nombre = nombre_completo.split("{")[0]
        posicion = re.search(r'position="(\d+)"', nombre_completo)
        if posicion:
            nombre = nombre + "[" + posicion.group(1) + "]"
        valores[nombre] = valores.get(nombre, 0) + float(valor)
    return valores

In [18]:
from concurrent.futures import ThreadPoolExecutor

def pedir(modelo, mensajes):
    #Temperatura 0 para que todas las configuraciones sean comparables
    r = requests.post(URL + "/v1/chat/completions", json={
        "model": modelo,
        "messages": mensajes,
        "temperature": 0,
        "max_tokens": MAX_TOKENS
    }, timeout=900)
    respuesta = r.json()
    if "choices" not in respuesta:
        raise RuntimeError(f"vLLM {r.status_code}: {respuesta}")
    return {"texto": respuesta["choices"][0]["message"]["content"], "tokens": respuesta["usage"]["completion_tokens"]}

def evaluar(nombre, modelo, concurrencia):
    #Calentamiento para que CUDA graphs y demás no cuenten en la medida
    pedir(modelo, prompts["es"][0])
    pedir(modelo, prompts["ca"][0])

    filas = []
    for idioma, lista in prompts.items():
        antes = metricas_spec()
        inicio = time.time()
        with ThreadPoolExecutor(max_workers=concurrencia) as grupo:
            resultados = list(grupo.map(lambda m: pedir(modelo, m), lista))
        segundos = time.time() - inicio
        diferencia = {k: v - antes.get(k, 0) for k, v in metricas_spec().items()}

        borradores = diferencia.get("vllm:spec_decode_num_drafts_total", 0)
        propuestos = diferencia.get("vllm:spec_decode_num_draft_tokens_total", 0)
        aceptados = diferencia.get("vllm:spec_decode_num_accepted_tokens_total", 0)
        tokens = sum(r["tokens"] for r in resultados)

        fila = {
            "modo": nombre,
            "concurrencia": concurrencia,
            "idioma": idioma,
            "tokens": tokens,
            "segundos": round(segundos, 1),
            "tok_s": round(tokens / segundos, 1),
            "longitud_aceptacion": round(1 + aceptados / borradores, 2) if borradores else None,
            "tasa_aceptacion": round(aceptados / propuestos, 3) if propuestos else None,
            "por_posicion": {k.split("[")[1][:-1]: v for k, v in diferencia.items() if "per_pos" in k},
            "textos": [r["texto"] for r in resultados]
        }
        filas.append(fila)
        print(nombre, "| c =", concurrencia, "|", idioma, "|", fila["tok_s"], "tok/s | aceptación:", fila["longitud_aceptacion"])
    return filas

In [19]:
#Rehago la selección saltando las prompts demasiado largas (dejo sitio para los 512 tokens de respuesta)
LIMITE = 8192 - MAX_TOKENS
prompts = {"es": [], "ca": [], "eu": [], "gl": []}
saltadas = Counter()

with open("/workspace/data/alia_eval_prompts.jsonl", encoding="utf-8") as archivo:
    for linea in archivo:
        conversacion = json.loads(linea)
        idioma = conversacion["lang"]
        if idioma not in prompts or len(prompts[idioma]) >= N_POR_IDIOMA:
            continue
        mensajes = prompt_de(conversacion)
        ids = tokenizador_objetivo.apply_chat_template(mensajes, add_generation_prompt=True, tokenize=True, return_dict=False)
        longitud = len(ids)
        if longitud > LIMITE:
            saltadas[idioma] += 1
            continue
        prompts[idioma].append(mensajes)

print({idioma: len(lista) for idioma, lista in prompts.items()})
print("Saltadas por largas:", dict(saltadas))

{'es': 20, 'ca': 20, 'eu': 20, 'gl': 20}
Saltadas por largas: {'es': 1}


In [ ]:
#Mido solo las configuraciones que faltan (las ya medidas están en comparacion.json)


ruta = "/workspace/resultados/comparacion.json"
resultados = json.load(open(ruta, encoding="utf-8")) if os.path.exists(ruta) else []
hechos = {f["modo"] for f in resultados}
print("Ya medidas:", hechos)

for nombre, config in CONFIGURACIONES.items():
    if nombre in hechos:
        continue
    try:
        servidor, modelo = arrancar(nombre, config)
    except RuntimeError as error:
        print(error)
        !tail -n 20 /workspace/vllm_{nombre}.log
        continue

    resultados += evaluar(nombre, modelo, 1)
    resultados += evaluar(nombre, modelo, 8)
    parar(servidor)

    #Guardo lo que llevo por si algo falla más adelante
    with open("/workspace/resultados/comparacion.json", "w", encoding="utf-8") as archivo:
        json.dump(resultados, archivo, ensure_ascii=False, indent=2)

Ya medidas: {'eagle3_ep2', 'base', 'draft_2b_base_gptq'}
vLLM listo: draft_2b_instruct_gptq
draft_2b_instruct_gptq | c = 1 | es | 104.8 tok/s | aceptación: 2.39
draft_2b_instruct_gptq | c = 1 | ca | 96.4 tok/s | aceptación: 2.44
draft_2b_instruct_gptq | c = 1 | eu | 89.6 tok/s | aceptación: 2.27
draft_2b_instruct_gptq | c = 1 | gl | 95.8 tok/s | aceptación: 2.27
draft_2b_instruct_gptq | c = 8 | es | 579.6 tok/s | aceptación: 2.41
draft_2b_instruct_gptq | c = 8 | ca | 558.4 tok/s | aceptación: 2.4
draft_2b_instruct_gptq | c = 8 | eu | 543.5 tok/s | aceptación: 2.23
draft_2b_instruct_gptq | c = 8 | gl | 593.2 tok/s | aceptación: 2.26
vLLM listo: base_7b_instruct_v1
base_7b_instruct_v1 | c = 1 | es | 159.7 tok/s | aceptación: None
base_7b_instruct_v1 | c = 1 | ca | 155.2 tok/s | aceptación: None
base_7b_instruct_v1 | c = 1 | eu | 149.2 tok/s | aceptación: None
base_7b_instruct_v1 | c = 1 | gl | 160.1 tok/s | aceptación: None
base_7b_instruct_v1 | c = 8 | es | 537.4 tok/s | aceptación: Non

In [47]:
import pandas as pd

#Cada configuración se compara con su propio modelo sin aceleración
ref = {n: (n if c["tipo"] == "base" else c.get("referencia", "base")) for n, c in CONFIGURACIONES.items()}

tabla = pd.DataFrame([{k: v for k, v in f.items() if k not in ("textos", "por_posicion")} for f in resultados])
tok_s = tabla.set_index(["modo", "concurrencia", "idioma"])["tok_s"]
tabla["speedup"] = [round(f.tok_s / tok_s[(ref.get(f.modo, "base"), f.concurrencia, f.idioma)], 2) for f in tabla.itertuples()]

print(tabla.pivot_table(index=["concurrencia", "idioma"], columns="modo", values="speedup"))
print()
print(tabla[tabla.speedup.notna() & ~tabla.modo.isin([n for n, c in CONFIGURACIONES.items() if c["tipo"] == "base"])]
      .pivot_table(index=["concurrencia", "idioma"], columns="modo", values="longitud_aceptacion"))

modo                 base  base_7b_instruct_v1  draft_2b_base_gptq  \
concurrencia idioma                                                  
1            ca       1.0                  1.0                0.65   
             es       1.0                  1.0                0.66   
             eu       1.0                  1.0                0.62   
             gl       1.0                  1.0                0.62   
8            ca       1.0                  1.0                0.67   
             es       1.0                  1.0                0.66   
             eu       1.0                  1.0                0.70   
             gl       1.0                  1.0                0.70   

modo                 draft_2b_instruct_gptq  draft_2b_instruct_gptq_v1  \
concurrencia idioma                                                      
1            ca                        0.57                       0.60   
             es                        0.63                       0.53   
   

In [48]:
#Aceptación por posición (1 petición a la vez): cuánto acierta cada draft en el 1º, 2º y 3º token
for fila in resultados:
    if fila["modo"] != "base" and fila["concurrencia"] == 1:
        print(fila["modo"], fila["idioma"], fila["por_posicion"])

eagle3_ep2 es {'0': 1579.0, '1': 1039.0, '2': 676.0}
eagle3_ep2 ca {'0': 1451.0, '1': 1028.0, '2': 734.0}
eagle3_ep2 eu {'0': 1476.0, '1': 996.0, '2': 681.0}
eagle3_ep2 gl {'0': 1538.0, '1': 1034.0, '2': 710.0}
draft_2b_base_gptq es {'0': 1348.0, '1': 959.0, '2': 685.0}
draft_2b_base_gptq ca {'0': 1263.0, '1': 884.0, '2': 640.0}
draft_2b_base_gptq eu {'0': 1387.0, '1': 941.0, '2': 679.0}
draft_2b_base_gptq gl {'0': 1532.0, '1': 1066.0, '2': 756.0}
draft_2b_instruct_gptq es {'0': 1491.0, '1': 1054.0, '2': 771.0}
draft_2b_instruct_gptq ca {'0': 1376.0, '1': 987.0, '2': 716.0}
draft_2b_instruct_gptq eu {'0': 1381.0, '1': 937.0, '2': 658.0}
draft_2b_instruct_gptq gl {'0': 1438.0, '1': 980.0, '2': 679.0}
base_7b_instruct_v1 es {}
base_7b_instruct_v1 ca {}
base_7b_instruct_v1 eu {}
base_7b_instruct_v1 gl {}
draft_2b_instruct_gptq_v1 es {'0': 537.0, '1': 295.0, '2': 175.0}
draft_2b_instruct_gptq_v1 ca {'0': 368.0, '1': 228.0, '2': 147.0}
draft_2b_instruct_gptq_v1 eu {'0': 205.0, '1': 112.0, '

In [49]:
#Comprobación: con cualquier draft las respuestas deberían salir iguales que sin aceleración (temperatura 0)
textos = {(f["modo"], f["idioma"]): f["textos"] for f in resultados if f["concurrencia"] == 1}

for nombre, config in CONFIGURACIONES.items():
    if config["tipo"] == "base":
        continue
    referencia = ref[nombre]
    for idioma in prompts:
        if (nombre, idioma) in textos and (referencia, idioma) in textos:
            iguales = sum(a == b for a, b in zip(textos[(referencia, idioma)], textos[(nombre, idioma)]))
            print(nombre, "vs", referencia, "|", idioma, "| idénticas:", iguales, "/", len(textos[(referencia, idioma)]))

eagle3_ep2 vs base | es | idénticas: 20 / 20
eagle3_ep2 vs base | ca | idénticas: 20 / 20
eagle3_ep2 vs base | eu | idénticas: 20 / 20
eagle3_ep2 vs base | gl | idénticas: 20 / 20
draft_2b_base_gptq vs base | es | idénticas: 6 / 20
draft_2b_base_gptq vs base | ca | idénticas: 8 / 20
draft_2b_base_gptq vs base | eu | idénticas: 9 / 20
draft_2b_base_gptq vs base | gl | idénticas: 3 / 20
draft_2b_instruct_gptq vs base | es | idénticas: 20 / 20
draft_2b_instruct_gptq vs base | ca | idénticas: 20 / 20
draft_2b_instruct_gptq vs base | eu | idénticas: 20 / 20
draft_2b_instruct_gptq vs base | gl | idénticas: 20 / 20
draft_2b_instruct_gptq_v1 vs base_7b_instruct_v1 | es | idénticas: 9 / 20
draft_2b_instruct_gptq_v1 vs base_7b_instruct_v1 | ca | idénticas: 14 / 20
draft_2b_instruct_gptq_v1 vs base_7b_instruct_v1 | eu | idénticas: 15 / 20
draft_2b_instruct_gptq_v1 vs base_7b_instruct_v1 | gl | idénticas: 9 / 20


In [ ]:
#Guardo y subo todo al repo de HF (carpeta eval/) para no perderlo al terminar el pod
tabla.to_csv("/workspace/resultados/comparacion.csv", index=False)
!nvidia-smi --query-gpu=name --format=csv,noheader > /workspace/resultados/gpu.txt
!cp /workspace/vllm_*.log /workspace/resultados/
!./vllm/bin/hf upload usuario/salamandra-7b-eagle3-alia /workspace/resultados eval --private

Found 6 files to upload
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ████████████████████  -
  Committing  ████████████████████  6 / 6 ✓
✓ Uploaded
  url: https://huggingface.co/gabmerlo/salamandra-7b-eagle3-alia/commit/6f71813dc821de267b4c7b6c401fd5ac798fbffc
